# Setup

In [ ]:
import torch
import numpy as np
import pandas as pd
import os
import json 
from os.path import dirname
pd.set_option("display.max_columns", None)
import random

torch.manual_seed(0)
torch.cuda.manual_seed(0)
random.seed(0)
np.random.seed(0)

from torch_geometric.data.hetero_data import HeteroData

torch.serialization.add_safe_globals([HeteroData])

from torch_geometric.loader import DataLoader

In [ ]:
# Paths and global variables
DATASET         = f"bpi_2012"

ROOT_PATH       = f"{dirname(os.getcwd())}/ISG"
# ROOT_PATH       = f"drive/MyDrive/Thesis" 
PROCESSED_PATH  = f"{ROOT_PATH}/data/datasets/processed/{DATASET}"
GRAPHS_PATH     = f"{ROOT_PATH}/data/datasets/graphs/{DATASET}"
MODEL_PATH      = f"{ROOT_PATH}/data/datsets/models/{DATASET}"

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

PATIENCE = 10
NUM_EPOCHS = 200
TOT_TRIALS = 20

In [ ]:
# Extract columns 
with open(f"{ROOT_PATH}/data/dataset_features.json", 'r') as file:
    dataset_info = json.load(file)[DATASET]

categorical_columns = dataset_info["categorical"]
real_value_columns = dataset_info["numerical"]

tab_all = pd.read_csv(f"{PROCESSED_PATH}/{DATASET}_processed_all.csv")

list_unique = {k : list(tab_all[k].unique()) for k in categorical_columns}

idx_to_activity = {i: name for i, name in enumerate(list_unique["Activity"])} # Index-activity map

In [ ]:
# Choose outputs
outputcat  = {k: len(list_unique[k]) for k in categorical_columns if k in ("Activity", "org:resource")}
outputreal = [k for k in real_value_columns if k == "time:timestamp"]

# +1 adds the end node
outputcat["Activity"] = outputcat["Activity"] + 1

In [ ]:
# Load graphs
X_TRAIN = torch.load(f"{GRAPHS_PATH}/train_set.pt", weights_only=False)
X_VALID = torch.load(f"{GRAPHS_PATH}/validation_set.pt", weights_only=False)
X_TEST  = torch.load(f"{GRAPHS_PATH}/test_set.pt", weights_only=False)

In [ ]:
edge_types = set()
node_types = set()
for i in range(len(X_TRAIN)):
    n, edge_type = X_TRAIN[i].metadata()
    for x in n:
        node_types.add(x)
    for x in edge_type:
        edge_types.add(x)
for i in range(len(X_VALID)):
    n, edge_type = X_VALID[i].metadata()
    for x in n:
        node_types.add(x)
    for x in edge_type:
        edge_types.add(x)
for i in range(len(X_TEST)):
    n, edge_type = X_TEST[i][0].metadata()
    for x in n:
        node_types.add(x)
    for x in edge_type:
        edge_types.add(x)

NODE_TYPES = list(node_types)
EDGE_TYPES = list(edge_types)

# Hyperparameter Optimization

In [ ]:
from ax.service.managed_loop import optimize
from torch_geometric.nn import (
    HeteroConv,
    GATv2Conv,global_mean_pool
)
from torch.nn import (
    ModuleList,
    Module,
    Linear,
    ModuleDict,
  )
from typing_extensions import Self

from torcheval.metrics.functional import multiclass_accuracy  
import torch.nn as nn
from copy import deepcopy
from tqdm.notebook import tqdm
from time import time

from torch.nn.functional import l1_loss
from torcheval.metrics.functional import multiclass_f1_score 

In [ ]:
class HGNN(Module):

    def __init__(self, output_cat, output_real, nodes_relations, parameters) -> Self:
        super().__init__()

        # Hyperparameters
        hid = parameters["hid"]
        layers = parameters["layers"]
        aggregation = parameters["aggregation"]

        # Set model outputs
        self.output_cat = output_cat
        self.output_real = output_real

        # Create convolutional layers
        self.convs = ModuleList()
        for _ in range(layers):
            # For each layer create a convolutional step (GATv2Conv) for each relationship type
            conv = HeteroConv(
                {
                    relation: (
                        GATv2Conv(
                            (-1, -1), hid, 1, False, add_self_loops=False, residual=False
                        )
                    )
                    for relation in nodes_relations
                },
                aggr=aggregation,
            )

            self.convs.append(conv)

        self.FC = ModuleDict()

        # Create a linear layer based on the output type
        for k in output_cat:
            self.FC[k] = Linear(hid, output_cat[k]) # Logits
        for k in output_real:
            self.FC[k] = Linear(hid, 1)             # Number


    def forward(self, batch):
        """
        Each convolutional layer:
        Takes the current node features (x_dict, a dict of node_type → tensor)
        Passes messages along edges (edge_index_dict, a dict of relation → edge list)
        Updates every node's embedding based on its neighbors
        Applies ReLU activation to introduce non-linearity
        """
        x_dict = batch.x_dict  # local copy

        for conv in self.convs:
            x_dict = conv(x_dict, batch.edge_index_dict)
            x_dict = {key: x.relu() for key, x in x_dict.items()}

        output = {}

        for k in self.output_cat:
            output[k] = global_mean_pool(x_dict[k], batch[k].batch)
            output[k] = self.FC[k](output[k])
        for k in self.output_real:
            output[k] = global_mean_pool(x_dict[k], batch[k].batch)
            output[k] = self.FC[k](output[k]).reshape(1, -1)[0]

        return output

In [ ]:
def train_hgnn(config, output_cat, output_real):
    print(config)

    # Instantiate the HGNN
    net = HGNN(
        parameters=config,
        output_cat=output_cat,
        output_real=output_real,
        nodes_relations=EDGE_TYPES,
    )
    net = net.to(device)

    # Asssign loss functions to the categories
    losses = {}
    for k in output_cat:
        losses[k] = nn.CrossEntropyLoss()
    for k in output_real:
        losses[k] = nn.L1Loss()

    # Data loading (automatical batches splitting!!)
    # In this case the DataLoader takes a batch of separate graph objects and merges them into one big disconnected graph.
    train_loader = DataLoader(X_TRAIN, batch_size=config["batch_size"], 
                          shuffle=True, num_workers=8, pin_memory=True)
    valid_loader = DataLoader(X_VALID, batch_size=config["batch_size"], 
                          shuffle=True, num_workers=8, pin_memory=True)
    
    # Adam optimizer + scheduler
    optimizer = torch.optim.Adam(
        net.parameters(), lr=config["lr"]
    )
    lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,mode="min", patience=int(PATIENCE/2)
    )
    
    # Trackers
    best_model = None
    best_loss = 0
    best_f1 = 0
    patience = PATIENCE
    pat_count = 0

    torch.cuda.empty_cache()
    training_times = []
    
    
    # TRAINING
    for epoch in tqdm(range(0, NUM_EPOCHS)):
        net.train()

        # == Inner training step ==
        for _, x in enumerate(train_loader):
            x = x.to(device)
            labels = x.y
            optimizer.zero_grad()
            
            # Forward pass!!
            outputs = net(x)

            # Compute each loss individually then total it
            losses_step = {k: losses[k](outputs[k], labels[k]) for k in losses}
            total_loss = 0
            for k in losses_step:
                total_loss += losses_step[k]

            # Compute gradients via backpropagation
            total_loss.backward()
            # Update parameters 
            optimizer.step()

        
        # Setup
        running_total_loss = []
        predictions_categorical = {k: [] for k in output_cat}
        target_categorical = {k: [] for k in output_cat}
        avg_MAE = {k : [] for k in output_real}
        prediction_numerical = {k: [] for k in output_real}
        target_numerical = {k: [] for k in output_real}
        

        # == Validation step ==
        net.eval()
        with torch.no_grad():
            for i, x in enumerate(valid_loader):
                x = x.to(device)
                labels = x.y
                
                # Forward pass
                outputs = net(x)

                # Compute loss
                losses_step = {k: losses[k](outputs[k], labels[k]) for k in losses}
                running_total_loss.append(sum(list(losses_step.values())))
            
                # INDENTING IT FORWARD BUT I DON'T KNOW IF THIS IS CORRECT
                for k in output_cat:
                    predictions_categorical[k].append(
                            torch.argmax(torch.softmax(outputs[k], dim=1), 1)
                    )
                    target_categorical[k].append(labels[k])
                
                
                for k in output_real:
                    prediction_numerical[k].append(
                        outputs[k]
                    )
                    target_numerical[k].append(
                        labels[k]
                    )

        
        # Mean loss in validation step, used to handle learning rate
        val_loss = sum(running_total_loss) / len(running_total_loss)
        lr_scheduler.step(val_loss)
        
        """
        torch.cat concatenates a list of tensors along dimension 0. 
        The pattern here is: during the loop you append per-batch tensors to a list, 
        then after the loop you join them into one big tensor covering the whole 
        validation set — standard practice in PyTorch evaluation. 
        """
        for k in predictions_categorical:
            predictions_categorical[k] = torch.cat(predictions_categorical[k])
            target_categorical[k] = torch.cat(target_categorical[k])
                
        for k in prediction_numerical:
            prediction_numerical[k] = torch.cat(prediction_numerical[k])
            target_numerical[k] = torch.cat(target_numerical[k])
    
        
       
        # == Metrics computation ==

        # averages the F1 score of each class equally,        
        macro_f1s = {
            k : multiclass_f1_score(
                predictions_categorical[k],
                target_categorical[k].squeeze(),
                num_classes=output_cat[k],
                average='macro'
            ) 
            for k in output_cat
        }
        f1_activity = macro_f1s["Activity"]
        
        
        # fraction of correct predictions
        accuracy = {
                k: multiclass_accuracy(
                    predictions_categorical[k],
                    target_categorical[k].squeeze(),
                    num_classes=output_cat[k],
                )
                for k in output_cat
            }
        
        # MAE recomputation
        MAE = {
            k: l1_loss(prediction_numerical[k], target_numerical[k]).item()
            for k in output_real
        }


        # == Metrics check ==
        if epoch == 0:
            best_model = deepcopy(net)
            best_loss = val_loss
            best_f1 = f1_activity
            print("/"*10)
            print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
            print("Acc", accuracy)
            print("F1s", macro_f1s)
            print("MAE", MAE)
            print(f"Patience {pat_count}/{patience}, val loss {val_loss} current_lr {lr_scheduler.get_last_lr()}, curr_best_activity_F1 {best_f1}")
        else:
            
            if val_loss < best_loss:
                best_loss = val_loss
                best_model = deepcopy(net)
                pat_count = 0
            if best_f1 < f1_activity:
                best_f1 = f1_activity

            print("/"*10)
            print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
            print("Acc", accuracy)
            print("F1s", macro_f1s)
            print("MAE", MAE)
            print(f"Patience {pat_count}/{patience}, val loss {val_loss} current_lr {lr_scheduler.get_last_lr()}, curr_best_activity_F1 {best_f1}")
            if pat_count == patience:
                return ({"valid_loss" : best_loss.item()} , best_model)
        pat_count += 1


    print("Acc", accuracy)
    print("F1s", macro_f1s)
    print("MAE", MAE)
    print(f"Patience {pat_count}/{patience}, val loss {val_loss} current_lr {lr_scheduler.get_last_lr()}, curr_best_activity_F1 {best_f1}")
    return ({"valid_loss" : best_loss.item()} , best_model)
   

In [ ]:
import logging

logging.getLogger("root").setLevel(logging.ERROR)

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
# Hyperparameters optimization (Ax)
def train_evaluate(config):
    res, _ = train_hgnn(config, output_cat=outputcat, output_real=outputreal)
    return res

best_parameters, values, experiment, model = optimize(
    parameters=[
        {"name": "hid", "type": "choice", "values": [128, 256, 512], "value_type": "int", "is_ordered": True, "sort_values": False},
        {"name": "batch_size", "type": "choice", "values": [256, 512, 1024], "value_type": "int", "is_ordered": True, "sort_values": False},
        {"name": "layers", "type": "choice", "values": [2,4], "value_type": "int", "is_ordered" : True, "sort_values":False},
        {"name": "lr", "type": "range", "bounds": [1e-4, 1e-2], "value_type": "float", "log_scale": True},
        {"name": "aggregation", "type" : "choice", "values" :["sum", "mean", "max"], "value_type" : "str"},
    ],
  
    evaluation_function=train_evaluate,
    objective_name='valid_loss',
    arms_per_trial=1,
    minimize = True,
    random_seed = 123,
    total_trials = TOT_TRIALS
)

print(best_parameters)
means, covariances = values
print(means)
print(experiment)

In [ ]:
from ax.service.utils.report_utils import exp_to_df

results = exp_to_df(experiment)
results = results.sort_values(by="valid_loss")
print(results)

if not os.path.isdir(f"{ROOT_PATH}/results/{DATASET}"):
    os.mkdir(f"{ROOT_PATH}/results/{DATASET}")

results.to_csv(f"{ROOT_PATH}/results/{DATASET}/hyp_params_search.csv", sep=",", index=False)
with open(f"{ROOT_PATH}/results/{DATASET}/best_parameters.csv", "w") as f:
    json.dump(best_parameters, f, indent=4)